In [ ]:
# Import necessary libraries
import ehtim as eh
import numpy as np
import runGenDIReCT
import torch

# Set device for PyTorch
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

# Define model parameters and paths
model_name = 'SR1_M87_2021_108_hilo_hops_netcal_StokesI_225'
model_path = 'models/saved_models/'+model_name+'.pt'
encoder_path = 'models/saved_models/Autoencoders/DIReCT_v2_AEweights.pt'
uvfits_path = 'uvfits/SR1_M87_2021_108_hilo_hops_netcal_StokesI.uvfits'
psize = 1.7044214966184275e-11 
imgdim = 64
baseid = 0

# Initialize the GenDIReCT model
model = runGenDIReCT.GenDIReCT(model_path, encoder_path, uvfits_path, device=device, baseid=baseid, imgdim=imgdim, psize=psize)
img = eh.image.make_empty(imgdim, imgdim*psize, 0, 0)
img = eh.image.load_fits('Images/s_gauss.fits')

# Trained model reconstructs image, useObs=True to use observed data
model.image(img, N_images=1024, useObs=True,
            plot_clusters=False, verbose=True, plot=False, evaluate=False)

# Save reconstructed image, if desired
final_img = model.convmodel().squeeze().detach().cpu().numpy()
np.save('results/reconstruction.npy', final_img)



In [ ]:
# Aggregate results from multiple runs and perform image-domain feature analysis/ring fitting
# In the default example, there are 50 reconstructions which have already been aligned

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import median_abs_deviation
import data.ringFitting as rf
import ehtim as eh

filename = 'results/M87_2021_108_225_sample.npy'
imgs = np.load(filename)
median_img = np.median(imgs, axis=0)
median_abs_dev = median_abs_deviation(imgs, axis=0)

fig, axs = plt.subplots(1, 3, figsize=(12, 4))
[ax.set_axis_off() for ax in axs]
fig.subplots_adjust(wspace=0, hspace=0)
fontsize=12
axs[0].imshow(median_img, cmap='inferno', interpolation='gaussian')
axs[0].set_title('Median Image', fontsize=fontsize)
axs[1].imshow(median_abs_dev, cmap='afmhot', interpolation='gaussian')
axs[1].set_title('Median Absolute Deviation Image', fontsize=fontsize)

threshold = 1e-5
median_abs_dev[median_abs_dev < threshold] = threshold  # Avoid division by near-zero
axs[2].contourf(median_img/median_abs_dev, levels=np.linspace(3, 15, 5), cmap='bone', alpha=1, origin='upper', extend='both')
axs[2].set_title('Signal-to-Noise Ratio Image', fontsize=fontsize)

load_fits = 'Images/s_crescent.fits'
model_img = eh.image.load_fits(load_fits)
model_img._imdict['I'] = median_img.flatten()

ringfitter = rf.RingFit(model_img, interp_factor=2, blur_uas=0, total_flux=1)
ringfitter.run_all(verbose=True)

height_ratio = [6,3]
fig, ax = plt.subplots(2,1, figsize=(np.sum(height_ratio),height_ratio[0]), height_ratios=height_ratio)
plt.subplots_adjust(hspace=0.0, wspace=0.0)
ringfitter.plot_all(ax[0], ax[1], cmap='Greys', ratio=height_ratio, save_path=None)


